In [1]:
# ==============================================================================
# CELL 0: Package Installation & Version Verification
# ==============================================================================
!pip install -q --upgrade \
    "transformers>=4.44.0" \
    "peft>=0.12.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=0.33.0" \
    "pyarrow>=14.0.0" \
    "datasets>=2.20.0"

import sys
import torch
import transformers
import peft
import bitsandbytes as bnb
import accelerate

if hasattr(torch.autograd, "graph") and hasattr(torch.autograd.graph, "set_warn_on_accumulate_grad_stream_mismatch"):
    torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False)

print("=" * 70)
print("STELLAR-ARM 10-SHARD ABLATION & TRAINING ENVIRONMENT VERIFICATION")
print("=" * 70)
print(f"Python Version:       {sys.version.split()[0]}")
print(f"PyTorch Version:      {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"Transformers:         {transformers.__version__}")
print(f"PEFT:                 {peft.__version__}")
print(f"BitsAndBytes:         {bnb.__version__}")
print(f"Accelerate:           {accelerate.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()} ({torch.cuda.device_count()} GPUs detected)")
print("=" * 70)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
STELLAR-ARM 10-SHARD ABLATION & TRAINING ENVIRONMENT VERIFICATION
Python Version:       3.13.16
PyTorch Version:      2.11.0+cu130 (CUDA: 13.0)
Transformers:         5.22.2
PEFT:                 0.21.2
BitsAndBytes:         0.50.2
Accelerate:           1.15.0
CUDA Available:       True (2 GPUs detected)


In [2]:
# ==============================================================================
# CELL 1: Ablation Mode Switch, Stage-Wise LR Schedule & Hardware Audit
# ==============================================================================
import os
import gc
import time
import math
import json
import random
import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Tuple, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------------------------
# 1. TOGGLE ABLATION MODE HERE:
#   - "SCALAR_RM"                -> Pure Bradley-Terry Scalar Baseline (No 6D supervision, No DAG, No Veto)
#   - "INDEPENDENT_6D"           -> Diagonal 6D Multi-Head Baseline (No DAG, No C1 Monotonicity, No A1 Veto, No C2 CF Loss)
#   - "STELLAR_NO_VETO"          -> STELLAR without A1 LRSV Veto (Tests linear compensation vs. gatekeeper veto)
#   - "STELLAR_NO_CF_LOSS"       -> STELLAR without C2 Counterfactual Activation Patching Loss (w_consist=0.0)
#   - "STELLAR_IN_BACKBONE_ONLY" -> STELLAR without B2 Post-Backbone 2-Layer CausalDAGLayer Mixer
#   - "STELLAR_FULL"             -> Full Production STELLAR-ARM (B1 + B2 + C1 + A1 + C2)
# ------------------------------------------------------------------------------
SHARD_ID = 1
EXPERIMENT_MODE = "SCALAR_RM"

# Optional speed toggle: 1.0 = full 7,500 pairs (469 steps, ~9.5h); 0.5 = stratified 3,750 pairs (235 steps, ~4.8h)
SUBSET_FRACTION = 1.0

# Set EVAL_ONLY_DIR to a mounted checkpoint path (e.g. your finished STELLAR_FULL/best path)
# if you want to skip training and immediately evaluate it on arm_test_benchmark.parquet in 5 minutes!
EVAL_ONLY_DIR = None

PREV_SHARD_BEST_DIR = None if SHARD_ID == 1 else "AUTO"
MID_SHARD_RESUME_DIR = None

SHARD_LR_SCHEDULE = {
    1:  {"lora_lr": 1.5e-4, "head_lr": 5.0e-4, "warmup_ratio": 0.06, "stage": "Shard 1: Cold-Start Foundation"},
    2:  {"lora_lr": 7.5e-5, "head_lr": 2.0e-4, "warmup_ratio": 0.04, "stage": "Shard 2: Broad Domain Expansion"},
    3:  {"lora_lr": 7.5e-5, "head_lr": 2.0e-4, "warmup_ratio": 0.04, "stage": "Shard 3: Broad Domain Expansion"},
    4:  {"lora_lr": 6.0e-5, "head_lr": 1.5e-4, "warmup_ratio": 0.04, "stage": "Shard 4: Broad Domain Expansion"},
    5:  {"lora_lr": 4.5e-5, "head_lr": 1.2e-4, "warmup_ratio": 0.03, "stage": "Shard 5: Mid-Stage Consolidation"},
    6:  {"lora_lr": 4.0e-5, "head_lr": 1.0e-4, "warmup_ratio": 0.03, "stage": "Shard 6: Mid-Stage Consolidation"},
    7:  {"lora_lr": 3.5e-5, "head_lr": 8.5e-5, "warmup_ratio": 0.03, "stage": "Shard 7: Mid-Stage Consolidation"},
    8:  {"lora_lr": 2.5e-5, "head_lr": 6.5e-5, "warmup_ratio": 0.03, "stage": "Shard 8: High-Precision SOTA Polish"},
    9:  {"lora_lr": 2.0e-5, "head_lr": 5.0e-5, "warmup_ratio": 0.03, "stage": "Shard 9: High-Precision SOTA Polish"},
    10: {"lora_lr": 1.5e-5, "head_lr": 4.0e-5, "warmup_ratio": 0.03, "stage": "Shard 10: Final SOTA Convergence"},
}

ACTIVE_LR_CFG = SHARD_LR_SCHEDULE[SHARD_ID]

EXPERIMENT_CONFIGS = {
    "STELLAR_FULL": {
        "associative": True,
        "use_dag_mixer": True,
        "use_monotonic": True,
        "use_lrsv_veto": True,
        "w_pref": 1.0,
        "w_dim": 4.0,
        "w_consist": 2.0,
        "scalar_only_mode": False,
        "description": "STELLAR-ARM Production Architecture (B1 DAG Bias + B2 Causal DAG Head + C1 Monotonicity + A1 LRSV Veto + C2 CF Loss)",
    },
    "STELLAR_IN_BACKBONE_ONLY": {
        "associative": True,
        "use_dag_mixer": False,
        "use_monotonic": True,
        "use_lrsv_veto": True,
        "w_pref": 1.0,
        "w_dim": 4.0,
        "w_consist": 2.0,
        "scalar_only_mode": False,
        "description": "STELLAR In-Backbone Only Ablation (B1 + C1 + A1 + C2 without post-backbone B2 mixer)",
    },
    "STELLAR_NO_VETO": {
        "associative": True,
        "use_dag_mixer": True,
        "use_monotonic": True,
        "use_lrsv_veto": False,
        "w_pref": 1.0,
        "w_dim": 4.0,
        "w_consist": 2.0,
        "scalar_only_mode": False,
        "description": "STELLAR without A1 LRSV Veto (Linear logit aggregation; tests non-compensatory gatekeeper)",
    },
    "STELLAR_NO_CF_LOSS": {
        "associative": True,
        "use_dag_mixer": True,
        "use_monotonic": True,
        "use_lrsv_veto": True,
        "w_pref": 1.0,
        "w_dim": 4.0,
        "w_consist": 0.0,
        "scalar_only_mode": False,
        "description": "STELLAR without C2 Counterfactual Activation Patching Loss (w_consist=0.0)",
    },
    "INDEPENDENT_6D": {
        "associative": False,
        "use_dag_mixer": False,
        "use_monotonic": False,
        "use_lrsv_veto": False,
        "w_pref": 1.0,
        "w_dim": 4.0,
        "w_consist": 0.0,
        "scalar_only_mode": False,
        "description": "Independent 6D Multi-Objective RM (Diagonal query mask, no DAG mixer, no C1/A1/C2)",
    },
    "SCALAR_RM": {
        "associative": False,
        "use_dag_mixer": False,
        "use_monotonic": False,
        "use_lrsv_veto": False,
        "w_pref": 1.0,
        "w_dim": 0.0,
        "w_consist": 0.0,
        "scalar_only_mode": True,
        "description": "Pure Bradley-Terry Scalar RM Baseline (Trained on pairwise preference only)",
    },
}

CFG = EXPERIMENT_CONFIGS[EXPERIMENT_MODE]
ACTIVE_CFG = CFG

assert torch.cuda.is_available(), "CUDA is required. Enable 2x T4 GPU in Kaggle settings."
num_gpus = torch.cuda.device_count()
HEAD_DEVICE = torch.device(f"cuda:{num_gpus - 1}") if num_gpus > 1 else torch.device("cuda:0")

print("=" * 85)
print(f"EXPERIMENT MODE : {EXPERIMENT_MODE} | SHARD {SHARD_ID}/10 ({ACTIVE_LR_CFG['stage']})")
print(f"Description     : {CFG['description']}")
print(f"Flags           : assoc={CFG['associative']}, B2_mixer={CFG['use_dag_mixer']}, C1_mono={CFG['use_monotonic']}, A1_veto={CFG['use_lrsv_veto']}")
print(f"Loss Weights    : w_pref={CFG['w_pref']}, w_dim={CFG['w_dim']}, w_consist={CFG['w_consist']} | Subset={SUBSET_FRACTION*100:.0f}%")
print(f"Shard LRs       : LoRA LR = {ACTIVE_LR_CFG['lora_lr']:.2e} | Head LR = {ACTIVE_LR_CFG['head_lr']:.2e}")
print("=" * 85)
for i in range(num_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory / (1024**3):.2f} GiB | CC: {p.major}.{p.minor}")

torch.cuda.empty_cache()
gc.collect()

SEED = 42 + SHARD_ID
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MAX_LENGTH = 1024
MICRO_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
LEARNING_RATE_LORA = ACTIVE_LR_CFG["lora_lr"]
LEARNING_RATE_HEADS = ACTIVE_LR_CFG["head_lr"]
WARMUP_RATIO = ACTIVE_LR_CFG["warmup_ratio"]
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
NUM_EPOCHS = 1
EVAL_EVERY_STEPS = 100
LOG_EVERY_STEPS = 20
MAX_WALLCLOCK_SECONDS = 10.2 * 3600

CHECKPOINT_ROOT = f"/kaggle/working/checkpoints/{EXPERIMENT_MODE}"
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)

EXPERIMENT MODE : SCALAR_RM | SHARD 1/10 (Shard 1: Cold-Start Foundation)
Description     : Pure Bradley-Terry Scalar RM Baseline (Trained on pairwise preference only)
Flags           : assoc=False, B2_mixer=False, C1_mono=False, A1_veto=False
Loss Weights    : w_pref=1.0, w_dim=0.0, w_consist=0.0 | Subset=100%
Shard LRs       : LoRA LR = 1.50e-04 | Head LR = 5.00e-04
  GPU 0: Tesla T4 | VRAM: 14.56 GiB | CC: 7.5
  GPU 1: Tesla T4 | VRAM: 14.56 GiB | CC: 7.5


In [3]:
# ==============================================================================
# CELL 2: Dataset Auto-Discovery & Single 4-Bit NF4 Qwen3-8B QLoRA Initialization
# ==============================================================================
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

DATASET_DIR = "/kaggle/input/notebooks/pradeep01223/stellar-arm-shard-data-pipeline-s1"
if not os.path.exists(os.path.join(DATASET_DIR, "arm_train.parquet")):
    for root, _, files in os.walk("/kaggle/input"):
        if "arm_train.parquet" in files and "arm_test_benchmark.parquet" in files:
            DATASET_DIR = root
            break

TRAIN_PATH = os.path.join(DATASET_DIR, "arm_train.parquet")
VAL_PATH   = os.path.join(DATASET_DIR, "arm_val.parquet")
TEST_PATH  = os.path.join(DATASET_DIR, "arm_test_benchmark.parquet")

print(f"[DATASET] Shard {SHARD_ID} Train : {TRAIN_PATH}")
print(f"[DATASET] Frozen Global Val   : {VAL_PATH}")
print(f"[DATASET] Frozen Global Test  : {TEST_PATH}")

MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
if not os.path.exists(MODEL_PATH):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "config.json" in files and ("qwen" in root.lower()):
            if "stellar" not in root.lower():
                MODEL_PATH = root
                break

print(f"[BACKBONE] Loading tokenizer from: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or "<|endoftext|>"
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

max_mem = {0: "8.0GiB", 1: "12.0GiB"} if num_gpus >= 2 else {0: "14.5GiB"}

print(f"[BACKBONE] Loading 4-bit Qwen3-8B AutoModel across {num_gpus} GPU(s) (max_memory={max_mem})...")
base_model = AutoModel.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory=max_mem,
    attn_implementation="eager",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
base_model.enable_input_require_grads()
base_model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

peft_backbone = get_peft_model(base_model, lora_config)
peft_backbone.print_trainable_parameters()

for i in range(num_gpus):
    alloc = torch.cuda.memory_allocated(i) / (1024 ** 3)
    resv = torch.cuda.memory_reserved(i) / (1024 ** 3)
    print(f"  GPU {i} VRAM after Backbone Init -> Allocated: {alloc:.2f} GiB | Reserved: {resv:.2f} GiB")

[DATASET] Shard 1 Train : /kaggle/input/notebooks/pradeep01223/stellar-arm-shard-data-pipeline-s1/arm_train.parquet
[DATASET] Frozen Global Val   : /kaggle/input/notebooks/pradeep01223/stellar-arm-shard-data-pipeline-s1/arm_val.parquet
[DATASET] Frozen Global Test  : /kaggle/input/notebooks/pradeep01223/stellar-arm-shard-data-pipeline-s1/arm_test_benchmark.parquet
[BACKBONE] Loading tokenizer from: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


[BACKBONE] Loading 4-bit Qwen3-8B AutoModel across 2 GPU(s) (max_memory={0: '8.0GiB', 1: '12.0GiB'})...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 43,646,976 || all params: 7,612,052,480 || trainable%: 0.5734
  GPU 0 VRAM after Backbone Init -> Allocated: 4.46 GiB | Reserved: 5.71 GiB
  GPU 1 VRAM after Backbone Init -> Allocated: 1.37 GiB | Reserved: 1.46 GiB


In [4]:
# ==============================================================================
# CELL 3: Smart Truncation & Dynamic Left-Padded DataLoaders (Train, Val, & Test)
# ==============================================================================
class ARMPairDataset(Dataset):
    def __init__(self, parquet_path: str, subset_fraction: float = 1.0, seed: int = 43):
        df = pd.read_parquet(parquet_path)
        if 0.0 < subset_fraction < 1.0:
            df = (
                df.groupby("regime", group_keys=False)
                .apply(lambda x: x.sample(frac=subset_fraction, random_state=seed))
                .sample(frac=1.0, random_state=seed)
                .reset_index(drop=True)
            )
        self.df = df
        self.prompts = self.df["prompt"].tolist()
        self.chosen = self.df["chosen"].tolist()
        self.rejected = self.df["rejected"].tolist()
        self.target_c = [list(map(float, x)) for x in self.df["target_chosen"].tolist()]
        self.target_r = [list(map(float, x)) for x in self.df["target_rejected"].tolist()]
        self.label_mask = [list(map(float, x)) for x in self.df["label_mask"].tolist()]
        self.regimes = self.df["regime"].tolist()
        self.pair_ids = self.df["pair_id"].tolist() if "pair_id" in self.df.columns else [str(i) for i in range(len(self.df))]

    def __len__(self) -> int:
        return len(self.prompts)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return {
            "prompt": self.prompts[idx],
            "chosen": self.chosen[idx],
            "rejected": self.rejected[idx],
            "target_chosen": self.target_c[idx],
            "target_rejected": self.target_r[idx],
            "label_mask": self.label_mask[idx],
            "regime": self.regimes[idx],
            "pair_id": self.pair_ids[idx],
        }

def truncate_prompt_response(prompt: str, response: str, tok: AutoTokenizer, max_len: int = 1024, resp_reserve: int = 640) -> List[int]:
    p_ids = tok.encode(prompt, add_special_tokens=False)
    r_ids = tok.encode(response, add_special_tokens=False)
    if len(p_ids) + len(r_ids) <= max_len:
        return p_ids + r_ids
    r_keep = r_ids[-min(len(r_ids), resp_reserve):] if resp_reserve > 0 else []
    p_budget = max_len - len(r_keep)
    p_keep = p_ids[-p_budget:] if p_budget > 0 else []
    return p_keep + r_keep

def collate_arm_batch(batch: List[Dict[str, Any]], tok: AutoTokenizer) -> Dict[str, Any]:
    c_lists = [truncate_prompt_response(it["prompt"], it["chosen"], tok, MAX_LENGTH, 640) for it in batch]
    r_lists = [truncate_prompt_response(it["prompt"], it["rejected"], tok, MAX_LENGTH, 640) for it in batch]
    batch_max_len = max(max(len(ids) for ids in c_lists), max(len(ids) for ids in r_lists))
    pad_id = tok.pad_token_id

    def pad_and_mask(ids_list: List[List[int]]) -> Tuple[torch.Tensor, torch.Tensor]:
        b_size = len(ids_list)
        padded = torch.full((b_size, batch_max_len), pad_id, dtype=torch.long)
        mask = torch.zeros((b_size, batch_max_len), dtype=torch.long)
        for i, ids in enumerate(ids_list):
            pad_len = batch_max_len - len(ids)
            padded[i, pad_len:] = torch.tensor(ids, dtype=torch.long)
            mask[i, pad_len:] = 1
        return padded, mask

    c_ids, c_mask = pad_and_mask(c_lists)
    r_ids, r_mask = pad_and_mask(r_lists)

    return {
        "chosen_input_ids": c_ids,
        "chosen_attention_mask": c_mask,
        "rejected_input_ids": r_ids,
        "rejected_attention_mask": r_mask,
        "target_chosen": torch.tensor(np.array([it["target_chosen"] for it in batch], dtype=np.float32)),
        "target_rejected": torch.tensor(np.array([it["target_rejected"] for it in batch], dtype=np.float32)),
        "label_mask": torch.tensor(np.array([it["label_mask"] for it in batch], dtype=np.float32)),
        "regimes": [it["regime"] for it in batch],
        "pair_ids": [it["pair_id"] for it in batch],
    }

train_dataset = ARMPairDataset(TRAIN_PATH, subset_fraction=SUBSET_FRACTION, seed=SEED)
val_dataset   = ARMPairDataset(VAL_PATH, subset_fraction=1.0, seed=SEED)
test_dataset  = ARMPairDataset(TEST_PATH, subset_fraction=1.0, seed=SEED)

g_seed = torch.Generator()
g_seed.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    generator=g_seed,
    num_workers=0,
    pin_memory=False,
    collate_fn=lambda b: collate_arm_batch(b, tokenizer),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=lambda b: collate_arm_batch(b, tokenizer),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=lambda b: collate_arm_batch(b, tokenizer),
)

print(f"[DATALOADERS READY] Shard {SHARD_ID} Train pairs: {len(train_dataset):,} ({len(train_loader):,} micro-batches)")
print(f"                    Frozen Val pairs  : {len(val_dataset):,} ({len(val_loader):,} micro-batches)")
print(f"                    Frozen Test pairs : {len(test_dataset):,} ({len(test_loader):,} micro-batches)")

[DATALOADERS READY] Shard 1 Train pairs: 7,500 (3,750 micro-batches)
                    Frozen Val pairs  : 750 (375 micro-batches)
                    Frozen Test pairs : 750 (375 micro-batches)


In [5]:
# ==============================================================================
# CELL 4: Qwen3AssociativeRM (STELLAR-ARM + Full Ablation Toggles) & Hybrid Loss
# ==============================================================================
class CausalDAGLayer(nn.Module):
    def __init__(self, d_model: int = 512, nhead: int = 8, dim_feedforward: int = 1024, dropout: float = 0.05):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.gate_proj = nn.Linear(d_model * 2, d_model)
        nn.init.constant_(self.gate_proj.bias, 1.0)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, associative: bool = True) -> torch.Tensor:
        dev = z.device
        if associative:
            attn_mask = torch.triu(torch.ones((6, 6), dtype=torch.bool, device=dev), diagonal=1)
        else:
            attn_mask = ~torch.eye(6, dtype=torch.bool, device=dev)

        c, _ = self.self_attn(z, z, z, attn_mask=attn_mask, need_weights=False)
        g = torch.sigmoid(self.gate_proj(torch.cat([z, c], dim=-1)))
        z_mixed = (1.0 - g) * z + g * c
        z_mixed = torch.cat([z[:, :1, :], z_mixed[:, 1:, :]], dim=1)
        z_out = self.norm1(z_mixed)
        z_out = self.norm2(z_out + self.ffn(z_out))
        return torch.cat([z[:, :1, :], z_out[:, 1:, :]], dim=1)


class Qwen3AssociativeRM(nn.Module):
    DIMENSION_NAMES = [
        "D1_INSTRUCTION_ADHERENCE",
        "D2_RELEVANCE",
        "D3_COMPLETENESS",
        "D4_REASONING_VALIDITY",
        "D5_TASK_CORRECTNESS",
        "D6_COMMUNICATION_QUALITY",
    ]
    GATEKEEPER_INDICES = [0, 3, 4]  # D1 (Adherence/Safety), D4 (Reasoning), D5 (Correctness)

    def __init__(
        self,
        peft_backbone: nn.Module,
        associative: bool = True,
        use_monotonic: bool = True,
        use_lrsv_veto: bool = True,
        beta: float = 1.0,
        scalar_only_mode: bool = False,
        dag_hidden_size: int = 512,
        dag_num_layers: int = 2,
        dag_num_heads: int = 8,
        veto_kappa: float = 1.5,
        monotonic_tau: float = 0.5,
        monotonic_delta_tol: float = 0.20,
    ):
        super().__init__()
        self.backbone = peft_backbone
        self.associative = associative
        self.use_monotonic = use_monotonic
        self.use_lrsv_veto = use_lrsv_veto
        self.beta = beta
        self.scalar_only_mode = scalar_only_mode
        self.dag_hidden_size = dag_hidden_size
        self.veto_kappa = veto_kappa
        self.monotonic_tau = monotonic_tau
        self.monotonic_delta_tol = monotonic_delta_tol

        self.input_device = self.backbone.get_input_embeddings().weight.device
        self.head_device = HEAD_DEVICE
        self.hidden_size = self.backbone.config.hidden_size

        self.query_embeds = nn.Parameter(
            torch.randn(1, 6, self.hidden_size, dtype=torch.float32, device=self.head_device) * 0.02
        )

        self.dag_attn_bias = nn.Parameter(
            torch.zeros((6, 6), dtype=torch.float32, device=self.head_device)
        )
        prior = torch.tril(torch.ones((6, 6), dtype=torch.float32, device=self.head_device), diagonal=-1)
        prior[4, 3] = 2.5
        prior[4, 2] = 2.0
        prior[3, 2] = 1.5
        self.register_buffer("dag_prior", prior, persistent=False)

        self.dag_down_proj = nn.Linear(self.hidden_size, dag_hidden_size, dtype=torch.float32, device=self.head_device)
        self.dag_norm = nn.LayerNorm(dag_hidden_size, dtype=torch.float32, device=self.head_device)
        self.dag_layers = nn.ModuleList([
            CausalDAGLayer(dag_hidden_size, dag_num_heads, dag_hidden_size * 2, dropout=0.05).to(
                device=self.head_device, dtype=torch.float32
            )
            for _ in range(dag_num_layers)
        ])

        self.dim_heads = nn.ModuleList([
            nn.Linear(dag_hidden_size, 1, dtype=torch.float32, device=self.head_device)
            for _ in range(6)
        ])

        self.agg_logits = nn.Parameter(torch.zeros(6, dtype=torch.float32, device=self.head_device))
        self.veto_thresholds = nn.Parameter(
            torch.zeros(len(self.GATEKEEPER_INDICES), dtype=torch.float32, device=self.head_device)
        )
        self.veto_raw_lambda = nn.Parameter(
            torch.tensor(1.85, dtype=torch.float32, device=self.head_device)
        )

    def build_4d_attention_mask(self, token_mask: torch.Tensor, dtype: torch.dtype) -> torch.Tensor:
        B, L = token_mask.shape
        T = L + 6
        dev = token_mask.device

        causal_grid = torch.tril(torch.ones((T, T), dtype=torch.bool, device=dev))
        if not self.associative:
            causal_grid[L:, L:] = torch.eye(6, dtype=torch.bool, device=dev)

        query_padding = torch.ones((B, 6), dtype=token_mask.dtype, device=dev)
        full_mask_2d = torch.cat([token_mask, query_padding], dim=1)
        key_valid = full_mask_2d.bool().unsqueeze(1).unsqueeze(2)

        valid_attn = causal_grid.unsqueeze(0).unsqueeze(0) & key_valid
        mask_4d = torch.where(
            valid_attn,
            torch.tensor(0.0, dtype=dtype, device=dev),
            torch.tensor(-1e4, dtype=dtype, device=dev),
        )

        if self.associative and (not self.scalar_only_mode):
            L_b = token_mask.float().sum(dim=-1, keepdim=True).clamp(min=1.0)
            log_L = 0.5 * torch.log(L_b).view(B, 1, 1, 1).to(device=dev, dtype=torch.float32)

            strict_tril = torch.tril(torch.ones((6, 6), dtype=torch.bool, device=dev), diagonal=-1)
            gamma_sig = torch.sigmoid(self.dag_attn_bias.to(device=dev, dtype=torch.float32))
            prior_dev = self.dag_prior.to(device=dev, dtype=torch.float32)

            q_bias = (log_L * gamma_sig.unsqueeze(0).unsqueeze(0) + prior_dev.unsqueeze(0).unsqueeze(0))
            q_bias = torch.where(strict_tril.unsqueeze(0).unsqueeze(0), q_bias, torch.zeros_like(q_bias))
            mask_4d[:, :, L:, L:] = mask_4d[:, :, L:, L:] + q_bias.to(dtype=dtype)

        return mask_4d

    def forward_dag_head_only(self, last_6_hidden_fp32: torch.Tensor) -> Dict[str, torch.Tensor]:
        z = self.dag_norm(self.dag_down_proj(last_6_hidden_fp32))
        for layer in self.dag_layers:
            z = layer(z, associative=self.associative)

        raw_logits = [self.dim_heads[i](z[:, i, :]) for i in range(6)]
        raw_z = torch.cat(raw_logits, dim=-1)
        z_clamped = torch.clamp(raw_z, min=-15.0, max=15.0)

        if self.scalar_only_mode:
            dim_logits = z_clamped
            dim_scores = torch.sigmoid(dim_logits)
            agg_weights = F.softmax(self.agg_logits, dim=0)
            scalar_logit = (dim_logits * agg_weights).sum(dim=-1)
            return {
                "scalar_logit": scalar_logit,
                "dim_logits": dim_logits,
                "dim_scores": dim_scores,
                "agg_weights": agg_weights,
                "dag_states": z,
            }

        if self.associative and self.use_monotonic:
            z3 = z_clamped[:, 2:3]
            z4 = z_clamped[:, 3:4]
            tau_m = self.monotonic_tau
            z_prereq = (
                -tau_m * torch.logsumexp(torch.cat([-z3 / tau_m, -z4 / tau_m], dim=-1), dim=-1, keepdim=True)
                + tau_m * math.log(2.0)
            )
            z5_mono = torch.clamp(
                z_prereq + self.monotonic_delta_tol - F.softplus(-raw_z[:, 4:5]),
                min=-15.0,
                max=15.0,
            )
            dim_logits = torch.cat([z_clamped[:, :4], z5_mono, z_clamped[:, 5:]], dim=-1)
        else:
            dim_logits = z_clamped

        dim_scores = torch.sigmoid(dim_logits)
        agg_weights = F.softmax(self.agg_logits, dim=0)
        linear_term = self.beta * (dim_logits * agg_weights).sum(dim=-1)

        if self.use_lrsv_veto:
            gk_logits = dim_logits[:, self.GATEKEEPER_INDICES]
            kappa = self.veto_kappa
            veto_lambda = F.softplus(self.veto_raw_lambda)
            exponents = -kappa * (gk_logits - self.veto_thresholds.unsqueeze(0))
            zero_pad = torch.zeros((exponents.size(0), 1), dtype=exponents.dtype, device=exponents.device)
            veto_penalty = (veto_lambda / kappa) * torch.logsumexp(torch.cat([zero_pad, exponents], dim=-1), dim=-1)
            scalar_logit = linear_term - veto_penalty
        else:
            scalar_logit = linear_term

        return {
            "scalar_logit": scalar_logit,
            "dim_logits": dim_logits,
            "dim_scores": dim_scores,
            "agg_weights": agg_weights,
            "dag_states": z,
        }

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        output_attentions: bool = False,
    ) -> Dict[str, Any]:
        B, L = input_ids.shape
        input_ids = input_ids.to(self.input_device)
        attention_mask = attention_mask.to(self.input_device)

        tok_embeds = self.backbone.get_input_embeddings()(input_ids)
        input_dtype = tok_embeds.dtype

        q_exp = self.query_embeds.to(device=self.input_device, dtype=input_dtype).expand(B, -1, -1)
        inputs_embeds = torch.cat([tok_embeds, q_exp], dim=1)

        query_mask = torch.ones((B, 6), dtype=attention_mask.dtype, device=self.input_device)
        full_mask_2d = torch.cat([attention_mask, query_mask], dim=1)
        pos_ids = (full_mask_2d.long().cumsum(-1) - 1).clamp(min=0)
        mask_4d = self.build_4d_attention_mask(attention_mask, dtype=input_dtype)

        outputs = self.backbone(
            inputs_embeds=inputs_embeds,
            attention_mask=mask_4d,
            position_ids=pos_ids,
            output_attentions=output_attentions,
            output_hidden_states=False,
            return_dict=True,
        )

        last_6_hidden = outputs.last_hidden_state[:, -6:, :]
        last_6_hidden_fp32 = last_6_hidden.to(device=self.head_device, dtype=torch.float32)

        with torch.cuda.device(self.head_device):
            head_out = self.forward_dag_head_only(last_6_hidden_fp32)

        res = {
            "scalar_logit": head_out["scalar_logit"],
            "dim_logits": head_out["dim_logits"],
            "dim_scores": head_out["dim_scores"],
            "agg_weights": head_out["agg_weights"],
            "last_6_hidden": last_6_hidden_fp32,
        }
        if output_attentions:
            res["attentions"] = outputs.attentions
        return res

    def get_head_state_dict(self) -> Dict[str, Any]:
        return {
            "query_embeds": self.query_embeds.detach().cpu(),
            "dag_attn_bias": self.dag_attn_bias.detach().cpu(),
            "dag_down_proj": self.dag_down_proj.state_dict(),
            "dag_norm": self.dag_norm.state_dict(),
            "dag_layers": self.dag_layers.state_dict(),
            "dim_heads": self.dim_heads.state_dict(),
            "agg_logits": self.agg_logits.detach().cpu(),
            "veto_thresholds": self.veto_thresholds.detach().cpu(),
            "veto_raw_lambda": self.veto_raw_lambda.detach().cpu(),
            "associative": self.associative,
            "use_monotonic": self.use_monotonic,
            "use_lrsv_veto": self.use_lrsv_veto,
            "beta": self.beta,
            "scalar_only_mode": self.scalar_only_mode,
        }

    def load_head_state_dict(self, state: Dict[str, Any]):
        self.query_embeds.data.copy_(state["query_embeds"].to(self.head_device))
        self.agg_logits.data.copy_(state["agg_logits"].to(self.head_device))
        if "dag_attn_bias" in state:
            self.dag_attn_bias.data.copy_(state["dag_attn_bias"].to(self.head_device))
            self.dag_down_proj.load_state_dict(state["dag_down_proj"])
            self.dag_norm.load_state_dict(state["dag_norm"])
            if len(self.dag_layers) > 0 and "dag_layers" in state:
                try:
                    self.dag_layers.load_state_dict(state["dag_layers"])
                except Exception:
                    pass
            self.dim_heads.load_state_dict(state["dim_heads"])
            self.veto_thresholds.data.copy_(state["veto_thresholds"].to(self.head_device))
            self.veto_raw_lambda.data.copy_(state["veto_raw_lambda"].to(self.head_device))
        self.associative = state.get("associative", self.associative)
        self.use_monotonic = state.get("use_monotonic", self.use_monotonic)
        self.use_lrsv_veto = state.get("use_lrsv_veto", self.use_lrsv_veto)
        self.beta = state.get("beta", self.beta)
        self.scalar_only_mode = state.get("scalar_only_mode", self.scalar_only_mode)


def soft_cross_entropy_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    targets_clamped = torch.clamp(targets, min=1e-4, max=1.0 - 1e-4)
    bce_raw = F.binary_cross_entropy_with_logits(logits, targets_clamped, reduction="none")
    target_entropy = -(
        targets_clamped * torch.log(targets_clamped)
        + (1.0 - targets_clamped) * torch.log(1.0 - targets_clamped)
    )
    calibrated_error = F.relu(bce_raw - target_entropy)
    masked_error = calibrated_error * mask
    denom = mask.sum().clamp(min=1.0)
    return masked_error.sum() / denom


def compute_arm_loss(
    model: Qwen3AssociativeRM,
    out_chosen: Dict[str, torch.Tensor],
    out_rejected: Dict[str, torch.Tensor],
    target_chosen: torch.Tensor,
    target_rejected: torch.Tensor,
    label_mask: torch.Tensor,
    w_pref: float = 1.0,
    w_dim: float = 4.0,
    w_consist: float = 2.0,
    gamma_ace: float = 0.50,
) -> Dict[str, torch.Tensor]:
    dev = out_chosen["scalar_logit"].device
    with torch.cuda.device(dev):
        target_chosen = target_chosen.to(dev)
        target_rejected = target_rejected.to(dev)
        label_mask = label_mask.to(dev)

        logit_diff = out_chosen["scalar_logit"] - out_rejected["scalar_logit"]
        loss_pref = -F.logsigmoid(logit_diff).mean()
        pref_acc = (logit_diff.detach() > 0.0).float().mean()

        if w_dim > 0.0:
            loss_dim_c = soft_cross_entropy_with_logits(out_chosen["dim_logits"], target_chosen, label_mask)
            loss_dim_r = soft_cross_entropy_with_logits(out_rejected["dim_logits"], target_rejected, label_mask)
            loss_dim = 0.5 * (loss_dim_c + loss_dim_r)
        else:
            loss_dim = torch.tensor(0.0, dtype=torch.float32, device=dev)

        all_scores = torch.cat([out_chosen["dim_scores"], out_rejected["dim_scores"]], dim=0)
        r_reason = all_scores[:, 3]
        r_correct = all_scores[:, 4]

        if w_consist > 0.0 and model.associative and ("last_6_hidden" in out_chosen):
            h_c_det = out_chosen["last_6_hidden"].detach()
            h_r_live = out_rejected["last_6_hidden"]
            h_cf = torch.cat([h_c_det[:, :3, :], h_r_live[:, 3:4, :], h_c_det[:, 4:, :]], dim=1)

            cf_out = model.forward_dag_head_only(h_cf)
            r5_cf = cf_out["dim_scores"][:, 4]
            r5_c = out_chosen["dim_scores"][:, 4].detach()
            r4_c = out_chosen["dim_scores"][:, 3].detach()
            r4_r = out_rejected["dim_scores"][:, 3]

            valid_pair_mask = ((target_chosen[:, 3] - target_rejected[:, 3]) > 0.30).float()
            required_drop = gamma_ace * F.relu(r4_c - r4_r)
            ace_hinge = F.relu(r5_cf - r5_c + required_drop)
            loss_ace = (valid_pair_mask * (ace_hinge ** 2)).sum() / valid_pair_mask.sum().clamp(min=1.0)

            r_reason_all = torch.cat([r_reason, r4_r], dim=0)
            r_correct_all = torch.cat([r_correct, r5_cf], dim=0)
            loss_mono = (F.softplus(10.0 * (r_correct_all - r_reason_all - 0.05)) ** 2).mean()

            loss_consist = loss_ace + 0.5 * loss_mono
        else:
            loss_consist = torch.tensor(0.0, dtype=torch.float32, device=dev)

        violation_rate = ((r_correct.detach() - r_reason.detach()) > 0.10).float().mean()
        total_loss = (w_pref * loss_pref) + (w_dim * loss_dim) + (w_consist * loss_consist)

    return {
        "total_loss": total_loss,
        "loss_pref": loss_pref.detach(),
        "loss_dim": loss_dim.detach(),
        "loss_consist": loss_consist.detach(),
        "pref_acc": pref_acc,
        "consist_violation_rate": violation_rate,
    }

dag_layers_count = 2 if ACTIVE_CFG.get("use_dag_mixer", True) else 0
model = Qwen3AssociativeRM(
    peft_backbone=peft_backbone,
    associative=ACTIVE_CFG["associative"],
    use_monotonic=ACTIVE_CFG["use_monotonic"],
    use_lrsv_veto=ACTIVE_CFG["use_lrsv_veto"],
    beta=1.0,
    scalar_only_mode=ACTIVE_CFG["scalar_only_mode"],
    dag_hidden_size=512,
    dag_num_layers=dag_layers_count,
    dag_num_heads=8,
    veto_kappa=1.5,
    monotonic_tau=0.5,
    monotonic_delta_tol=0.20,
)
print(f"[MODEL INITIALIZED] Mode={EXPERIMENT_MODE} on {model.head_device} (DAG Layers={dag_layers_count})")

[MODEL INITIALIZED] Mode=SCALAR_RM on cuda:1 (DAG Layers=0)


In [6]:
# ==============================================================================
# CELL 5: Checkpoint Manager & Auto-Chaining Engine
# ==============================================================================
from peft import get_peft_model_state_dict, set_peft_model_state_dict

def _json_safe_default(obj):
    if isinstance(obj, (np.floating, float)):
        return float(obj)
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    return str(obj)

def save_checkpoint(
    rm_model: Qwen3AssociativeRM,
    opt: torch.optim.Optimizer,
    sched: Any,
    step: int,
    global_best_val_acc: float,
    shard_best_val_acc: float,
    metrics_history: List[Dict[str, Any]],
    tag: str = "latest",
):
    save_dir = os.path.join(CHECKPOINT_ROOT, tag)
    os.makedirs(save_dir, exist_ok=True)

    lora_state = get_peft_model_state_dict(rm_model.backbone)
    torch.save(lora_state, os.path.join(save_dir, "lora_weights.pt"))
    torch.save(rm_model.get_head_state_dict(), os.path.join(save_dir, "arm_heads.pt"))
    torch.save(
        {
            "shard_id": int(SHARD_ID),
            "step": int(step),
            "best_val_acc": float(global_best_val_acc),
            "shard_best_val_acc": float(shard_best_val_acc),
            "optimizer": opt.state_dict(),
            "scheduler": sched.state_dict(),
            "experiment_mode": EXPERIMENT_MODE,
        },
        os.path.join(save_dir, "trainer_state.pt"),
    )
    with open(os.path.join(save_dir, "metrics_log.json"), "w") as f:
        json.dump(metrics_history, f, indent=2, default=_json_safe_default)
    print(
        f"  [CHECKPOINT SAVED] -> {save_dir} (Shard {SHARD_ID}, Step {step} | "
        f"Shard Best: {float(shard_best_val_acc)*100:.2f}% | Global Best: {float(global_best_val_acc)*100:.2f}%)",
        flush=True,
    )

def load_weights_from_dir(rm_model: Qwen3AssociativeRM, ckpt_dir: str):
    set_peft_model_state_dict(
        rm_model.backbone,
        torch.load(os.path.join(ckpt_dir, "lora_weights.pt"), map_location="cpu", weights_only=False),
    )
    rm_model.load_head_state_dict(
        torch.load(os.path.join(ckpt_dir, "arm_heads.pt"), map_location=rm_model.head_device, weights_only=False)
    )
    print(f"  [LOADED WEIGHTS] Restored LoRA + Head weights from: {ckpt_dir}", flush=True)

In [7]:
# ==============================================================================
# CELL 6: Training Loop + Held-Out Test Benchmark (`arm_test_benchmark.parquet`)
# ==============================================================================
backbone_params = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
head_params = [
    p for n, p in model.named_parameters()
    if (not n.startswith("backbone.")) and p.requires_grad
]

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": LEARNING_RATE_LORA,  "weight_decay": WEIGHT_DECAY},
        {"params": head_params,     "lr": LEARNING_RATE_HEADS, "weight_decay": WEIGHT_DECAY},
    ]
)

total_micro_batches = len(train_loader) * NUM_EPOCHS
total_opt_steps = math.ceil(total_micro_batches / GRAD_ACCUM_STEPS)
warmup_steps = max(10, int(WARMUP_RATIO * total_opt_steps))

def lr_lambda(current_step: int) -> float:
    if current_step < warmup_steps:
        return float(current_step + 1) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_opt_steps - warmup_steps))
    return max(0.15, 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress))))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def evaluate_split(rm_model: Qwen3AssociativeRM, dataloader: DataLoader, split_label: str, step: int) -> Dict[str, Any]:
    rm_model.eval()
    total_pairs = 0
    total_correct = 0
    total_loss_accum = 0.0
    total_violations = 0
    total_responses = 0

    regime_correct = {}
    regime_total = {}
    dim_abs_err = [0.0 for _ in range(6)]
    dim_counts = [0 for _ in range(6)]

    for batch in dataloader:
        b_sz = batch["chosen_input_ids"].size(0)
        out_c = rm_model(batch["chosen_input_ids"], batch["chosen_attention_mask"])
        out_r = rm_model(batch["rejected_input_ids"], batch["rejected_attention_mask"])

        l_dict = compute_arm_loss(
            rm_model, out_c, out_r,
            batch["target_chosen"], batch["target_rejected"], batch["label_mask"],
            w_pref=ACTIVE_CFG["w_pref"], w_dim=ACTIVE_CFG["w_dim"], w_consist=ACTIVE_CFG["w_consist"]
        )
        total_loss_accum += float(l_dict["total_loss"].item()) * b_sz

        diffs = (out_c["scalar_logit"] - out_r["scalar_logit"]).detach().cpu().numpy()
        corrs = (diffs > 0.0)
        total_correct += int(corrs.sum())
        total_pairs += int(b_sz)

        for reg, c_flag in zip(batch["regimes"], corrs):
            regime_total[reg] = regime_total.get(reg, 0) + 1
            regime_correct[reg] = regime_correct.get(reg, 0) + int(c_flag)

        sc_c = out_c["dim_scores"].detach().cpu().numpy()
        sc_r = out_r["dim_scores"].detach().cpu().numpy()
        tc = batch["target_chosen"].numpy()
        tr = batch["target_rejected"].numpy()
        msk = batch["label_mask"].numpy()

        for i in range(b_sz):
            for d in range(6):
                if msk[i, d] > 0.5:
                    dim_abs_err[d] += float(abs(sc_c[i, d] - tc[i, d])) + float(abs(sc_r[i, d] - tr[i, d]))
                    dim_counts[d] += 2

        all_sc = np.concatenate([sc_c, sc_r], axis=0)
        viols = (all_sc[:, 4] > (all_sc[:, 3] + 0.10))
        total_violations += int(viols.sum())
        total_responses += int(len(all_sc))

    acc = float(total_correct) / float(max(1, total_pairs))
    loss_avg = float(total_loss_accum) / float(max(1, total_pairs))
    viol_pct = (float(total_violations) / float(max(1, total_responses))) * 100.0
    dim_mae = {
        Qwen3AssociativeRM.DIMENSION_NAMES[d]: round(float(dim_abs_err[d]) / float(max(1, dim_counts[d])), 4)
        for d in range(6)
    }
    avg_mae = round(float(np.mean(list(dim_mae.values()))), 4)
    regime_acc = {
        reg: round((float(regime_correct[reg]) / float(max(1, regime_total[reg]))) * 100.0, 2)
        for reg in sorted(regime_total.keys())
    }

    weights_np = F.softmax(rm_model.agg_logits, dim=0).detach().cpu().numpy()
    veto_lam = float(F.softplus(rm_model.veto_raw_lambda).item()) if rm_model.use_lrsv_veto else 0.0

    print("\n" + "=" * 92)
    print(f"[{split_label} @ MODE={EXPERIMENT_MODE} | SHARD {SHARD_ID} | STEP {step}/{total_opt_steps}]")
    print(f"  Loss: {loss_avg:.4f} | Overall Acc: {acc*100:.2f}% | Avg Dim MAE: {avg_mae:.4f} | Level-C Viol: {viol_pct:.2f}% | Veto Lambda: {veto_lam:.3f}")
    print(f"  Regime Acc (%): {regime_acc}")
    print(f"  Dim MAE:        {dim_mae}")
    print(f"  Agg Weights:    {[round(float(x), 4) for x in weights_np]}")
    print("=" * 92 + "\n", flush=True)

    rm_model.train()
    return {
        "split": split_label,
        "experiment_mode": EXPERIMENT_MODE,
        "shard_id": int(SHARD_ID),
        "step": int(step),
        "overall_acc_pct": round(float(acc) * 100.0, 2),
        "total_loss": round(float(loss_avg), 4),
        "consist_violation_pct": round(float(viol_pct), 2),
        "regime_acc": regime_acc,
        "dim_mae": dim_mae,
        "dim_mae_avg": avg_mae,
        "veto_lambda": round(float(veto_lam), 4),
    }

# ------------------------------------------------------------------------------
# Execute Training (Unless EVAL_ONLY_DIR is set)
# ------------------------------------------------------------------------------
metrics_history = []
global_best_val_acc = 0.0
shard_best_val_acc = 0.0

if EVAL_ONLY_DIR and os.path.exists(EVAL_ONLY_DIR):
    print(f"[EVAL-ONLY MODE] Loading existing checkpoint from {EVAL_ONLY_DIR}...")
    load_weights_from_dir(model, EVAL_ONLY_DIR)
    val_res = evaluate_split(model, val_loader, "FROZEN_VAL", step=total_opt_steps)
    test_res = evaluate_split(model, test_loader, "HELD_OUT_TEST_BENCHMARK", step=total_opt_steps)
else:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    session_start_time = time.time()
    opt_step = 0
    running_loss = running_pref = running_dim = running_consist = running_acc = running_viol = 0.0
    micro_count = 0

    print(f"\n[TRAINING START] Launching {EXPERIMENT_MODE} on Shard {SHARD_ID} for {total_opt_steps} optimizer steps...", flush=True)

    for mb_idx, batch in enumerate(train_loader, start=1):
        out_c = model(batch["chosen_input_ids"], batch["chosen_attention_mask"])
        out_r = model(batch["rejected_input_ids"], batch["rejected_attention_mask"])

        loss_dict = compute_arm_loss(
            model, out_c, out_r,
            batch["target_chosen"], batch["target_rejected"], batch["label_mask"],
            w_pref=ACTIVE_CFG["w_pref"],
            w_dim=ACTIVE_CFG["w_dim"],
            w_consist=ACTIVE_CFG["w_consist"],
        )

        scaled_loss = loss_dict["total_loss"] / GRAD_ACCUM_STEPS
        with torch.cuda.device(model.head_device):
            scaled_loss.backward()

        running_loss += float(loss_dict["total_loss"].item())
        running_pref += float(loss_dict["loss_pref"].item())
        running_dim += float(loss_dict["loss_dim"].item())
        running_consist += float(loss_dict["loss_consist"].item())
        running_acc += float(loss_dict["pref_acc"].item())
        running_viol += float(loss_dict["consist_violation_rate"].item())
        micro_count += 1

        is_step_boundary = (mb_idx % GRAD_ACCUM_STEPS == 0) or (mb_idx == total_micro_batches)
        if is_step_boundary:
            opt_step += 1
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            if opt_step % LOG_EVERY_STEPS == 0 or opt_step == 1:
                elapsed_h = (time.time() - session_start_time) / 3600.0
                cur_lora_lr = scheduler.get_last_lr()[0]
                print(
                    f"Shard {SHARD_ID:02d} | Step {opt_step:3d}/{total_opt_steps} | "
                    f"Loss: {running_loss/micro_count:.4f} "
                    f"(Pref: {running_pref/micro_count:.4f}, Dim: {running_dim/micro_count:.4f}, CF: {running_consist/micro_count:.4f}) | "
                    f"Acc: {(running_acc/micro_count)*100:.1f}% | "
                    f"Viol: {(running_viol/micro_count)*100:.2f}% | "
                    f"LR: {cur_lora_lr:.2e} | Elapsed: {elapsed_h:.2f}h",
                    flush=True,
                )
                running_loss = running_pref = running_dim = running_consist = running_acc = running_viol = 0.0
                micro_count = 0

            if (opt_step % EVAL_EVERY_STEPS == 0) or (opt_step == total_opt_steps):
                val_metrics = evaluate_split(model, val_loader, "VALIDATION", opt_step)
                metrics_history.append(val_metrics)
                cur_val_acc = val_metrics["overall_acc_pct"] / 100.0

                if cur_val_acc >= shard_best_val_acc:
                    shard_best_val_acc = cur_val_acc
                    save_checkpoint(model, optimizer, scheduler, opt_step, max(global_best_val_acc, cur_val_acc), shard_best_val_acc, metrics_history, tag="shard_best")

                if cur_val_acc >= (global_best_val_acc - 0.0025):
                    global_best_val_acc = max(global_best_val_acc, cur_val_acc)
                    save_checkpoint(model, optimizer, scheduler, opt_step, global_best_val_acc, shard_best_val_acc, metrics_history, tag="best")

                save_checkpoint(model, optimizer, scheduler, opt_step, global_best_val_acc, shard_best_val_acc, metrics_history, tag="latest")
                torch.cuda.empty_cache()

            if (time.time() - session_start_time) >= MAX_WALLCLOCK_SECONDS and opt_step < total_opt_steps:
                elapsed_h = (time.time() - session_start_time) / 3600.0
                print(f"\n[SAFE TIMEOUT] Reached {elapsed_h:.2f} hours at Step {opt_step}. Saving state and exiting cleanly!", flush=True)
                save_checkpoint(model, optimizer, scheduler, opt_step, global_best_val_acc, shard_best_val_acc, metrics_history, tag="latest")
                break

    # Reload best checkpoint and evaluate on Held-Out Test Benchmark (`arm_test_benchmark.parquet`)
    best_dir = os.path.join(CHECKPOINT_ROOT, "best")
    if os.path.exists(os.path.join(best_dir, "lora_weights.pt")):
        load_weights_from_dir(model, best_dir)
    test_res = evaluate_split(model, test_loader, "HELD_OUT_TEST_BENCHMARK", step=opt_step)

# Save final structured ablation report JSON
report_path = f"/kaggle/working/ablation_results_{EXPERIMENT_MODE}_shard{SHARD_ID}.json"
with open(report_path, "w") as f:
    json.dump(
        {
            "experiment_mode": EXPERIMENT_MODE,
            "shard_id": SHARD_ID,
            "subset_fraction": SUBSET_FRACTION,
            "best_val_acc_pct": round(float(shard_best_val_acc) * 100.0, 2),
            "held_out_test_benchmark": test_res,
            "validation_history": metrics_history,
        },
        f,
        indent=2,
        default=_json_safe_default,
    )
print(f"[ABLATION REPORT SAVED] -> {report_path}", flush=True)


[TRAINING START] Launching SCALAR_RM on Shard 1 for 469 optimizer steps...
Shard 01 | Step   1/469 | Loss: 0.7141 (Pref: 0.7141, Dim: 0.0000, CF: 0.0000) | Acc: 31.2% | Viol: 25.00% | LR: 1.07e-05 | Elapsed: 0.01h
Shard 01 | Step  20/469 | Loss: 0.5089 (Pref: 0.5089, Dim: 0.0000, CF: 0.0000) | Acc: 75.3% | Viol: 25.99% | LR: 1.12e-04 | Elapsed: 0.37h
Shard 01 | Step  40/469 | Loss: 0.2543 (Pref: 0.2543, Dim: 0.0000, CF: 0.0000) | Acc: 92.5% | Viol: 17.50% | LR: 1.50e-04 | Elapsed: 0.72h
Shard 01 | Step  60/469 | Loss: 0.2284 (Pref: 0.2284, Dim: 0.0000, CF: 0.0000) | Acc: 90.0% | Viol: 20.16% | LR: 1.48e-04 | Elapsed: 1.08h
Shard 01 | Step  80/469 | Loss: 0.2135 (Pref: 0.2135, Dim: 0.0000, CF: 0.0000) | Acc: 90.9% | Viol: 22.81% | LR: 1.45e-04 | Elapsed: 1.44h
Shard 01 | Step 100/469 | Loss: 0.2384 (Pref: 0.2384, Dim: 0.0000, CF: 0.0000) | Acc: 90.9% | Viol: 68.44% | LR: 1.40e-04 | Elapsed: 1.81h

[VALIDATION @ MODE=SCALAR_RM | SHARD 1 | STEP 100/469]
  Loss: 0.1983 | Overall Acc: 91.0